Additional uncertainty analysis of the frozen procedure

This analysis was added after the original final evaluation. It uses saved reserved-test labels, scores and flags to estimate approximate 95% bootstrap confidence intervals for AP, precision, recall and F1. The fitted model and threshold remain unchanged. Whole customer rows are resampled together, assuming independent records. This analysis does not test whether Random Forest outperforms logistic regression.

In [1]:
#Mount Drive and load the existing evidence
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import hashlib
import json
import platform
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import sklearn
from sklearn.metrics import average_precision_score

PROJECT = Path(
    "/content/drive/MyDrive/ML 2026-DS-77 Telco Customer Churn"
)

EVALUATION_DIR = PROJECT / "outputs" / "tables" / "final_evaluation"

PREDICTIONS_PATH = EVALUATION_DIR / "final_test_predictions.csv"
SUMMARY_PATH = EVALUATION_DIR / "final_test_summary.csv"

assert PREDICTIONS_PATH.exists(), PREDICTIONS_PATH
assert SUMMARY_PATH.exists(), SUMMARY_PATH

# These files contain the existing evaluation results.
# No model is loaded, trained or tuned.
predictions = pd.read_csv(PREDICTIONS_PATH)
original_summary = pd.read_csv(SUMMARY_PATH)

print("Prediction columns:", predictions.columns.tolist())
print("Test records:", len(predictions))

Mounted at /content/drive
Prediction columns: ['customerID', 'actual_churn', 'churn_score', 'high_risk_flag']
Test records: 1409


In [2]:
#Verify the predictions before calculating intervals
required_columns = [
    "customerID",
    "actual_churn",
    "churn_score",
    "high_risk_flag",
]

assert set(required_columns).issubset(predictions.columns)
assert not predictions[required_columns].isna().any().any()
assert predictions["customerID"].is_unique
assert len(predictions) == 1409

# Check values before converting them.
assert set(predictions["actual_churn"].unique()) == {0, 1}
assert set(predictions["high_risk_flag"].unique()).issubset({0, 1})

y = predictions["actual_churn"].to_numpy(dtype=int)
scores = predictions["churn_score"].to_numpy(dtype=float)
flags = predictions["high_risk_flag"].to_numpy(dtype=int)

assert np.isfinite(scores).all()
assert ((scores >= 0) & (scores <= 1)).all()
assert len(original_summary) == 1

threshold = float(original_summary.loc[0, "Threshold"])

# Verify that the saved flags follow the existing frozen rule.
assert np.array_equal(flags, (scores >= threshold).astype(int))
assert y.sum() == 374
assert flags.sum() == 519

METRICS = ["AP", "Precision", "Recall", "F1"]


def calculate_metrics(labels, risk_scores, risk_flags):
    """Calculate metrics without fitting or changing any procedure."""
    tp = np.sum((labels == 1) & (risk_flags == 1))
    fp = np.sum((labels == 0) & (risk_flags == 1))
    fn = np.sum((labels == 1) & (risk_flags == 0))

    # Fail explicitly if a bootstrap draw cannot support these metrics.
    if np.unique(labels).size != 2 or tp + fp == 0:
        raise ValueError(
            "A bootstrap draw lacks both classes or has no flagged records."
        )

    return np.array([
        average_precision_score(labels, risk_scores),
        tp / (tp + fp),
        tp / (tp + fn),
        2 * tp / (2 * tp + fp + fn),
    ])


point_estimates = calculate_metrics(y, scores, flags)

saved_estimates = original_summary.loc[
    0, ["Test AP", "Test precision", "Test recall", "Test F1"]
].to_numpy(dtype=float)

# Confirm that we reproduce the original reported results.
np.testing.assert_allclose(
    point_estimates,
    saved_estimates,
    rtol=0,
    atol=1e-10,
)

display(pd.DataFrame({
    "Metric": METRICS,
    "Original test estimate": point_estimates,
}).round(4))

print("Checks passed. Model and threshold remain unchanged.")

,Metric,Original test estimate
0,AP,0.6565
1,Precision,0.5491
2,Recall,0.7620
3,F1,0.6383


Checks passed. Model and threshold remain unchanged.


In [3]:
#Run the bootstrap analysis
N_BOOTSTRAPS = 10_000
BOOTSTRAP_SEED = 42

rng = np.random.default_rng(BOOTSTRAP_SEED)
n_customers = len(predictions)

bootstrap_values = np.empty((N_BOOTSTRAPS, len(METRICS)))

for repetition in range(N_BOOTSTRAPS):
    # Draw whole rows with replacement.
    # The same indices keep each label, score and flag together.
    indices = rng.integers(
        low=0,
        high=n_customers,
        size=n_customers,
    )

    bootstrap_values[repetition] = calculate_metrics(
        y[indices],
        scores[indices],
        flags[indices],
    )

    if (repetition + 1) % 2000 == 0:
        print(f"Completed {repetition + 1:,} bootstrap repetitions")

# Percentile intervals: central 95% of bootstrap metric values.
lower_bounds, upper_bounds = np.quantile(
    bootstrap_values,
    [0.025, 0.975],
    axis=0,
    method="linear",
)

confidence_intervals = pd.DataFrame({
    "Metric": METRICS,
    "Test estimate": point_estimates,
    "95% CI lower": lower_bounds,
    "95% CI upper": upper_bounds,
})

display(confidence_intervals.round(4))

#Do not change the seed or method to obtain narrower or more favourable intervals. Their purpose is to describe uncertainty.

Completed 2,000 bootstrap repetitions
Completed 4,000 bootstrap repetitions
Completed 6,000 bootstrap repetitions
Completed 8,000 bootstrap repetitions
Completed 10,000 bootstrap repetitions


,Metric,Test estimate,95% CI lower,95% CI upper
0,AP,0.6565,0.6070,0.7087
1,Precision,0.5491,0.5063,0.5927
2,Recall,0.7620,0.7191,0.8042
3,F1,0.6383,0.6014,0.6748


In [4]:
#Save the results and analysis record
# A separate output location preserves the original evaluation files.
OUTPUT_DIR = EVALUATION_DIR / "uncertainty_analysis"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


def sha256_file(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()


run_record = {
    "analysis": "Post-evaluation uncertainty analysis of frozen procedure",
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "prediction_file": str(PREDICTIONS_PATH),
    "prediction_file_sha256": sha256_file(PREDICTIONS_PATH),
    "summary_file_sha256": sha256_file(SUMMARY_PATH),
    "test_records": n_customers,
    "recorded_churners": int(y.sum()),
    "frozen_threshold": threshold,
    "bootstrap_method": "Ordinary paired-row percentile bootstrap",
    "resampling_unit": "Customer row",
    "bootstrap_repetitions": N_BOOTSTRAPS,
    "random_seed": BOOTSTRAP_SEED,
    "confidence_level": 0.95,
    "quantile_method": "linear",
    "model_refitted": False,
    "threshold_reselected": False,
    "between_model_comparison": False,
    "assumption": "Independent customer records",
    "scope": (
        "Test-sample uncertainty conditional on the fitted procedure; "
        "does not include training or model-selection variability, "
        "temporal validity, or population representativeness."
    ),
    "python": platform.python_version(),
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "scikit_learn": sklearn.__version__,
}

# Timestamped outputs avoid overwriting earlier analysis runs.
run_tag = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")

confidence_intervals.to_csv(
    OUTPUT_DIR / f"test_confidence_intervals_{run_tag}.csv",
    index=False,
)

pd.DataFrame(
    bootstrap_values,
    columns=METRICS,
).to_csv(
    OUTPUT_DIR / f"bootstrap_metric_values_{run_tag}.csv",
    index=False,
)

with open(
    OUTPUT_DIR / f"uncertainty_run_record_{run_tag}.json",
    "x",
    encoding="utf-8",
) as file:
    json.dump(run_record, file, indent=2)

print("Saved uncertainty analysis in:", OUTPUT_DIR)

Saved uncertainty analysis in: /content/drive/MyDrive/ML 2026-DS-77 Telco Customer Churn/outputs/tables/final_evaluation/uncertainty_analysis
